# Retail Sales Data Cleaning with Python & Pandas

**Portfolio project | Google Colab ready**

This project takes deliberately messy retail order records through a reproducible cleaning pipeline, joins reference tables, handles dates and time zones, checks data quality, and produces analysis-ready outputs. It demonstrates practical pandas methods rather than trying to list every method that exists.

**Skills covered:** inspection, types, missing values (`dropna`, `fillna`, `ffill`, `bfill`, interpolation), duplicates, string `.str` methods, vectorized operations, `lambda`/`map`/`apply`, dates and time zones, `date_range`, joins/merges, groupby, reshaping, outlier checks, validation, and export.

> Run cells from top to bottom. The data is generated locally, so no external download is needed.


## 1. Setup and reproducibility

In [ ]:
import sys, platform, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
print("Python:", sys.version.split()[0], "| pandas:", pd.__version__, "| NumPy:", np.__version__)


## 2. Create realistic raw data

In a real project, replace this section with `pd.read_csv(...)`, `pd.read_excel(...)`, or a database extract. The intentional issues below let the cleaning steps be demonstrated every time.

In [ ]:
n = 240
customer_ids = [f"C{i:03d}" for i in range(1, 61)]
product_ids = [f"P{i:02d}" for i in range(1, 9)]
dates = pd.date_range("2025-01-01", periods=120, freq="D")
raw = pd.DataFrame({
    "order_id": [f"O{i:04d}" for i in range(1, n+1)],
    "customer_id": rng.choice(customer_ids, n).astype(object),
    "product_id": rng.choice(product_ids, n).astype(object),
    "customer_name": rng.choice(["  aLIce  johnson", "BOB SMITH", "ChArLiE  Lee", "dana brown", None], n),
    "email": rng.choice(["ALICE@example.com", " bob@example.com ", "not-an-email", None, "charlie@example.org"], n),
    "city": rng.choice(["new york", " NEW YORK ", "Mumbai", "mumbai ", "Delhi", None], n),
    "order_date": rng.choice(dates.strftime("%Y-%m-%d"), n).astype(object),
    "order_time": rng.choice(["09:15", "14:30", "18:05", "23:45", "bad-time"], n).astype(object),
    "quantity": rng.choice([1, 2, 3, 4, 5, None, "two"], n).astype(object),
    "unit_price": rng.choice([199.0, 499.0, 999.0, 1499.0, None, "unknown"], n).astype(object),
    "discount_pct": rng.choice([0, 5, 10, 15, None], n).astype(object),
    "status": rng.choice([" shipped ", "DELIVERED", "cancelled", "Pending", None], n),
    "channel": rng.choice(["web", "store", "app", "WEB", None], n),
})
# Inject whitespace, inconsistent identifiers, blanks, invalid values, duplicates, and a negative quantity.
raw.loc[raw.index[::17], "customer_id"] = raw.loc[raw.index[::17], "customer_id"].astype(str).str.lower()
raw.loc[raw.index[::19], "email"] = "   "
raw.loc[7, "quantity"] = -2
raw = pd.concat([raw, raw.iloc[[2, 5]]], ignore_index=True)
raw.head(8)


## 3. Initial inspection and data-quality baseline

Inspect shape, types, samples, uniqueness, missingness, and duplicate records before modifying anything. `info()` is useful for a compact structural summary.

In [ ]:
print("Shape:", raw.shape)
display(raw.head())
display(raw.sample(5, random_state=SEED))
raw.info()
display(raw.describe(include="all").T)
missing_before = raw.isna().sum().sort_values(ascending=False).rename("missing_count").to_frame()
missing_before["missing_pct"] = (missing_before["missing_count"] / len(raw) * 100).round(1)
display(missing_before)
print("Exact duplicate rows:", raw.duplicated().sum())
print("Distinct order IDs:", raw["order_id"].nunique())


## 4. Standardize text and identifiers

Use vectorized string operations: `.str.strip()`, `.str.lower()`, `.str.upper()`, `.str.title()`, `.str.replace()`, `.str.contains()`, `.str.startswith()`, `.str.len()`, `.str.split()`, and `.str.extract()`. These operate on an entire Series without a Python loop.

In [ ]:
df = raw.copy()
# Normalize blank-like values first.
text_cols = ["customer_id", "product_id", "customer_name", "email", "city", "order_date", "order_time", "status", "channel"]
for col in text_cols:
    df[col] = df[col].astype("string").str.strip()
    df[col] = df[col].replace("", pd.NA)
df["customer_id"] = df["customer_id"].str.upper()
df["product_id"] = df["product_id"].str.upper()
df["customer_name"] = df["customer_name"].str.replace(r"\s+", " ", regex=True).str.title()
df["email"] = df["email"].str.lower()
df["city"] = df["city"].str.replace(r"\s+", " ", regex=True).str.title()
df["status"] = df["status"].str.lower()
df["channel"] = df["channel"].str.lower()
# Examples of extraction and boolean string checks.
df["email_domain"] = df["email"].str.extract(r"@([a-zA-Z0-9.-]+)$", expand=False)
df["email_looks_valid"] = df["email"].str.contains(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", regex=True, na=False)
df["customer_name_length"] = df["customer_name"].str.len()
display(df[["customer_name", "email", "email_domain", "email_looks_valid", "city", "status", "channel"]].head(10))


## 5. Parse numeric values and dates safely

`errors="coerce"` turns invalid values into missing values so they can be counted and handled explicitly. Nullable `Int64` preserves integer values while allowing missing entries.

In [ ]:
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce").astype("Int64")
df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")
df["discount_pct"] = pd.to_numeric(df["discount_pct"], errors="coerce")
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce", format="%Y-%m-%d")
df["order_time_parsed"] = pd.to_datetime(df["order_time"], format="%H:%M", errors="coerce").dt.time
print(df.dtypes)
print("Unparseable quantity:", df["quantity"].isna().sum())
print("Unparseable unit_price:", df["unit_price"].isna().sum())
print("Unparseable time:", df["order_time_parsed"].isna().sum())


## 6. Missing values: measure, drop, fill, forward/back-fill, interpolate

There is no universal best imputation rule. Choose based on meaning and document it. Here we remove rows missing required keys, fill categorical fields with an explicit label, use the median for numeric fields, and show `ffill`, `bfill`, and interpolation on a separate ordered time series so they do not silently invent order-level facts.

In [ ]:
missing_report = df.isna().sum().sort_values(ascending=False).to_frame("count")
missing_report["pct"] = (missing_report["count"] / len(df) * 100).round(1)
display(missing_report)
# dropna examples: subset identifies required fields; thresh keeps rows with enough observed values.
required = df.dropna(subset=["order_id", "customer_id", "product_id", "order_date"]).copy()
print("After required-field dropna:", required.shape)
example_thresh = df.dropna(thresh=8)  # at least 8 non-null values across a row
print("Rows retained with thresh=8:", len(example_thresh))
# Impute numeric and categorical columns using explicit, reviewable rules.
clean = required.copy()
clean["quantity"] = clean["quantity"].fillna(clean["quantity"].median()).astype("Int64")
clean["unit_price"] = clean["unit_price"].fillna(clean["unit_price"].median())
clean["discount_pct"] = clean["discount_pct"].fillna(0)
for col in ["customer_name", "email", "city", "status", "channel"]:
    clean[col] = clean[col].fillna("Unknown")
# Demonstrate filling methods on a daily series (appropriate after sorting by time).
daily_demo = pd.Series([10.0, np.nan, np.nan, 16.0, np.nan], index=pd.date_range("2025-01-01", periods=5, freq="D"), name="metric")
display(pd.DataFrame({"original": daily_demo, "ffill": daily_demo.ffill(), "bfill": daily_demo.bfill(), "interpolate": daily_demo.interpolate()}))


## 7. Duplicates, invalid values, and business rules

`duplicated()` can detect exact or key-based duplicates; `drop_duplicates()` keeps the chosen record. Do not remove duplicate-looking records without deciding what makes a record unique.

In [ ]:
print("Exact duplicate rows before:", clean.duplicated().sum())
clean = clean.drop_duplicates(subset="order_id", keep="first").copy()
print("Rows after order_id dedupe:", len(clean))
# Business rule checks: quantities and prices cannot be negative; discount must be 0..100.
invalid_quantity = clean["quantity"] <= 0
invalid_price = clean["unit_price"] < 0
invalid_discount = ~clean["discount_pct"].between(0, 100)
print({"invalid_quantity": int(invalid_quantity.sum()), "invalid_price": int(invalid_price.sum()), "invalid_discount": int(invalid_discount.sum())})
# Exclude invalid records and preserve a copy for an audit count.
invalid_mask = invalid_quantity | invalid_price | invalid_discount
rejected_rows = clean.loc[invalid_mask].copy()
clean = clean.loc[~invalid_mask].copy()
print("Rejected rows:", len(rejected_rows), "| Clean rows:", len(clean))
# Filtering examples: loc, query, isin, between, nlargest, nsmallest.
shipped = clean.loc[clean["status"].isin(["shipped", "delivered"])]
recent = clean.query("quantity >= 2 and discount_pct <= 15")
display(clean.nlargest(5, "unit_price")[["order_id", "unit_price", "quantity"]])
display(clean.nsmallest(5, "unit_price")[["order_id", "unit_price", "quantity"]])


## 8. Vectorized features, lambda, map, and apply

Prefer vectorized Series arithmetic and NumPy selection for row-wise calculations. `map` is handy for one-column lookups; `apply` is flexible but can be slower on large data. Use a lambda for a concise transformation when it improves readability.

In [ ]:
# Vectorized arithmetic across whole columns.
clean["gross_amount"] = clean["quantity"] * clean["unit_price"]
clean["discount_amount"] = clean["gross_amount"] * clean["discount_pct"] / 100
clean["net_amount"] = clean["gross_amount"] - clean["discount_amount"]
# np.select is a vectorized multi-condition equivalent to chained row-level if statements.
conditions = [clean["net_amount"] >= 3000, clean["net_amount"] >= 1000]
choices = ["High", "Medium"]
clean["order_value_band"] = np.select(conditions, choices, default="Low")
# map category labels and lambda transformation.
channel_labels = {"web": "Online Web", "app": "Mobile App", "store": "Retail Store", "Unknown": "Unknown"}
clean["channel_label"] = clean["channel"].map(channel_labels).fillna("Other")
clean["customer_initials"] = clean["customer_name"].map(lambda x: "".join(part[0] for part in x.split() if part and x != "Unknown"))
# apply example: custom row-level classification, shown for learning (vectorized result above is preferred).
clean["amount_check"] = clean.apply(lambda row: "Review" if row["net_amount"] < 0 else "OK", axis=1)
# Other vectorized operations: round, clip, where, mask.
clean["net_amount"] = clean["net_amount"].round(2)
clean["discount_pct_capped"] = clean["discount_pct"].clip(0, 50)
clean["large_order_flag"] = clean["net_amount"].where(clean["net_amount"] >= 2500, 0).gt(0)
display(clean[["quantity", "unit_price", "gross_amount", "discount_pct", "net_amount", "order_value_band", "channel_label"]].head())


## 9. Dates, date_range, time zones, and time-series operations

The source date is a calendar date. We combine it with a parsed local time, localize it to the assumed source zone (`Asia/Kolkata`), then convert the instant to UTC. Localization assigns a zone to a naive timestamp; conversion changes the displayed zone while preserving the instant.

In [ ]:
clean["order_year"] = clean["order_date"].dt.year
clean["order_month"] = clean["order_date"].dt.month
clean["month_name"] = clean["order_date"].dt.month_name()
clean["weekday"] = clean["order_date"].dt.day_name()
clean["year_month"] = clean["order_date"].dt.to_period("M").astype("string")
# Date range: create a complete daily calendar to expose dates with no orders.
calendar = pd.DataFrame({"order_date": pd.date_range(clean["order_date"].min(), clean["order_date"].max(), freq="D")})
# Build local timestamps only where both date and time parsed.
valid_time = clean["order_time_parsed"].notna()
clean["local_timestamp"] = pd.NaT
clean.loc[valid_time, "local_timestamp"] = pd.to_datetime(
    clean.loc[valid_time, "order_date"].dt.strftime("%Y-%m-%d") + " " + clean.loc[valid_time, "order_time"].astype("string"), errors="coerce"
)
clean["local_timestamp"] = pd.to_datetime(clean["local_timestamp"]).dt.tz_localize("Asia/Kolkata", ambiguous="NaT", nonexistent="NaT")
clean["utc_timestamp"] = clean["local_timestamp"].dt.tz_convert("UTC")
# Timedelta example: time since the start of the order date.
clean["minutes_after_midnight"] = (clean["local_timestamp"].dt.hour * 60 + clean["local_timestamp"].dt.minute).astype("Float64")
print("Local zone:", clean["local_timestamp"].dt.tz, "| UTC zone:", clean["utc_timestamp"].dt.tz)
display(clean[["order_date", "order_time", "local_timestamp", "utc_timestamp", "weekday", "year_month"]].head())


### Daily aggregation, reindexing, resampling

`resample()` requires a datetime index. Reindexing against the generated calendar exposes missing dates; fill a reporting metric with zero only when that is the intended meaning.

In [ ]:
daily_sales = (clean.set_index("order_date").resample("D").agg(
    orders=("order_id", "nunique"), revenue=("net_amount", "sum"), units=("quantity", "sum")
))
daily_sales = daily_sales.reindex(calendar["order_date"], fill_value=0)
daily_sales.index.name = "order_date"
monthly_sales = clean.set_index("order_date").resample("MS")["net_amount"].sum().rename("revenue")
display(daily_sales.head())
display(monthly_sales)


## 10. Joins and merges

`merge` supports inner, left, right, and outer joins, plus cross joins. `validate` checks expected key relationships; `indicator=True` shows where rows came from. `DataFrame.join()` is convenient for index-based joins. `merge_asof()` matches each row to the nearest prior/next key and is useful for time-aligned lookup data.

In [ ]:
customers = pd.DataFrame({
    "customer_id": customer_ids,
    "segment": rng.choice(["Consumer", "Small Business", "Enterprise"], len(customer_ids)),
    "loyalty_tier": rng.choice(["Bronze", "Silver", "Gold"], len(customer_ids)),
})
products = pd.DataFrame({
    "product_id": product_ids,
    "product_name": ["Notebook", "Backpack", "Pen Set", "Desk Lamp", "Water Bottle", "Headphones", "Mouse", "Keyboard"],
    "category": ["Stationery", "Accessories", "Stationery", "Home", "Home", "Electronics", "Electronics", "Electronics"],
})
# Primary analytical join: many orders belong to one customer/product.
clean = clean.merge(customers, on="customer_id", how="left", validate="many_to_one")
clean = clean.merge(products, on="product_id", how="left", validate="many_to_one")
# Demonstrate each join type on small key tables.
a = pd.DataFrame({"key": [1, 2, 3], "left_value": ["A", "B", "C"]})
b = pd.DataFrame({"key": [2, 3, 4], "right_value": [20, 30, 40]})
join_examples = {
    "inner": a.merge(b, on="key", how="inner"),
    "left": a.merge(b, on="key", how="left"),
    "right": a.merge(b, on="key", how="right"),
    "outer": a.merge(b, on="key", how="outer", indicator=True),
    "cross": a.head(2).merge(b.head(2), how="cross"),
}
for name, result in join_examples.items():
    print(f"\n{name.upper()} JOIN")
    display(result)
# Index-based DataFrame.join().
indexed_customers = customers.set_index("customer_id")
orders_joined_by_index = clean.set_index("customer_id").join(indexed_customers[["segment"]], rsuffix="_lookup")
# merge_asof example: match each transaction to latest available FX rate at or before timestamp.
transactions = clean.dropna(subset=["local_timestamp"])[["order_id", "local_timestamp", "net_amount"]].sort_values("local_timestamp")
rates = pd.DataFrame({"local_timestamp": pd.to_datetime(["2025-01-01", "2025-02-01", "2025-03-01"]).tz_localize("Asia/Kolkata"), "fx_rate": [1.0, 1.01, 1.02]}).sort_values("local_timestamp")
asof_demo = pd.merge_asof(transactions, rates, on="local_timestamp", direction="backward")
display(asof_demo.head())


## 11. Groupby, aggregation, pivot, and reshape

After cleaning, summarize business performance. `agg` computes several measures; `pivot_table` turns groups into a matrix; `melt` returns wide data to tidy long form; `get_dummies` encodes categories for modeling.

In [ ]:
summary = (clean.groupby(["category", "channel_label"], dropna=False)
           .agg(orders=("order_id", "nunique"), units=("quantity", "sum"), revenue=("net_amount", "sum"), avg_order_value=("net_amount", "mean"))
           .reset_index().sort_values("revenue", ascending=False))
display(summary)
pivot = pd.pivot_table(clean, index="category", columns="channel_label", values="net_amount", aggfunc="sum", fill_value=0, margins=True)
display(pivot)
wide = clean.pivot_table(index="year_month", columns="category", values="net_amount", aggfunc="sum", fill_value=0).reset_index()
long = wide.melt(id_vars="year_month", var_name="category", value_name="revenue")
display(long.head())
encoded = pd.get_dummies(clean[["channel_label", "category"]], dtype=int)
display(encoded.head())


## 12. Outlier review and useful DataFrame methods

Outlier flags are investigation aids, not automatic deletion rules. This uses the IQR rule for net amount. Also see `sort_values`, `value_counts`, `nunique`, `rename`, `assign`, `reset_index`, `astype`, and `convert_dtypes`.

In [ ]:
q1, q3 = clean["net_amount"].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
clean["amount_outlier_flag"] = ~clean["net_amount"].between(lower, upper)
print(f"IQR bounds: {lower:.2f} .. {upper:.2f}; flagged: {clean['amount_outlier_flag'].sum()}")
display(clean["status"].value_counts(dropna=False).to_frame("count"))
print("Unique customers:", clean["customer_id"].nunique())
clean = clean.convert_dtypes()
print(clean.dtypes)


## 13. Quality checks and cleaning impact

A portfolio-quality workflow makes assumptions visible and checks its outputs. These assertions are lightweight data contracts; adapt thresholds to the real business rules.

In [ ]:
quality_checks = {
    "order_id unique": clean["order_id"].is_unique,
    "required keys present": clean[["order_id", "customer_id", "product_id", "order_date"]].notna().all().all(),
    "quantity positive": clean["quantity"].gt(0).all(),
    "unit price nonnegative": clean["unit_price"].ge(0).all(),
    "discount between 0 and 100": clean["discount_pct"].between(0, 100).all(),
    "net amount nonnegative": clean["net_amount"].ge(0).all(),
}
for check, passed in quality_checks.items():
    print(("PASS" if passed else "FAIL"), "-", check)
assert all(quality_checks.values()), "A data-quality rule failed; inspect the cleaning logic."
impact = pd.DataFrame({
    "stage": ["Raw input", "After required-field handling", "After deduplication and business rules", "Rejected for business rules"],
    "rows": [len(raw), len(required), len(clean), len(rejected_rows)]
})
display(impact)


## 14. Export clean data and analysis tables

In Colab, files are written to the current runtime and can be downloaded from the Files panel. Uncomment the download cell if you want an immediate browser download.

In [ ]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
clean.to_csv(OUTPUT_DIR / "cleaned_retail_orders.csv", index=False)
summary.to_csv(OUTPUT_DIR / "sales_summary.csv", index=False)
daily_sales.to_csv(OUTPUT_DIR / "daily_sales.csv")
# Excel does not store timezone-aware datetimes. Keep their timezone/offset in ISO text form for Excel only.
clean_for_excel = clean.copy()
for col in ["local_timestamp", "utc_timestamp"]:
    if col in clean_for_excel.columns:
        clean_for_excel[col] = clean_for_excel[col].astype("string")
with pd.ExcelWriter(OUTPUT_DIR / "retail_cleaning_outputs.xlsx") as writer:
    clean_for_excel.to_excel(writer, sheet_name="Clean Orders", index=False)
    summary.to_excel(writer, sheet_name="Sales Summary", index=False)
    daily_sales.to_excel(writer, sheet_name="Daily Sales")
print("Saved files:", sorted(p.name for p in OUTPUT_DIR.iterdir()))
# In Google Colab, uncomment to download a file:
# from google.colab import files
# files.download("outputs/retail_cleaning_outputs.xlsx")


## 15. Optional quick visual

A simple plot makes the final result easier to present. `matplotlib` is used through pandas plotting.

In [ ]:
ax = monthly_sales.plot(kind="bar", figsize=(9, 4), color="#2E86AB", title="Monthly Net Sales")
ax.set_xlabel("Month")
ax.set_ylabel("Net sales")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 16. Complete pandas practice checklist (Parts 4–12)

This section explicitly demonstrates every item in the requested syllabus, using the cleaned `clean` DataFrame. Each cell is runnable after the earlier project cells.


### Part 4 — Selecting and filtering data

Single/multiple columns, `loc`, `iloc`, conditional logic with AND/OR/NOT, `isin`, `between`, `query`, `filter`, and `select_dtypes`.

In [ ]:
# Single column (Series) and multiple columns (DataFrame)
order_ids = clean["order_id"]
order_subset = clean[["order_id", "customer_id", "net_amount"]]
# loc is label-based; iloc is integer-position-based.
loc_example = clean.loc[clean["net_amount"].ge(1000), ["order_id", "net_amount"]].head()
iloc_example = clean.iloc[:5, :4]
# AND (&), OR (|), NOT (~): parenthesize each condition.
condition_and = clean.loc[(clean["quantity"] >= 2) & (clean["discount_pct"] > 0)]
condition_or = clean.loc[(clean["status"] == "shipped") | (clean["status"] == "delivered")]
condition_not = clean.loc[~clean["status"].isin(["cancelled", "Unknown"])]
chosen_cities = clean.loc[clean["city"].isin(["Mumbai", "Delhi"])]
mid_value = clean.loc[clean["net_amount"].between(500, 2500, inclusive="both")]
query_example = clean.query("quantity >= 2 and net_amount < 3000")
# filter() selects labels by items, substring, or regex; axis=1 means columns.
filter_items = clean.filter(items=["order_id", "customer_id", "net_amount"])
filter_regex = clean.filter(regex="amount|price", axis=1)
numeric_columns = clean.select_dtypes(include="number")
print("Examples:", len(order_ids), order_subset.shape, loc_example.shape, iloc_example.shape, condition_and.shape, condition_or.shape, condition_not.shape, chosen_cities.shape, mid_value.shape, query_example.shape, filter_items.shape, filter_regex.shape, numeric_columns.shape)


### Part 5 — Cleaning and preprocessing methods

`isnull` and `isna` are aliases. `notnull`/`notna` identify observed values. This cell demonstrates `drop`, `rename`, `replace`, and missing-data checks without changing the main analysis frame. The notebook's earlier cells already demonstrate `dropna`, `fillna`, forward/back fill, duplicates, conversion, and IQR outlier flags.

In [ ]:
missing_isnull = clean.isnull().sum()
missing_isna = clean.isna().sum()
observed_notnull = clean["email"].notnull()
observed_notna = clean["email"].notna()
# replace values in a copy (dictionary replacement can target individual columns).
replace_demo = clean[["status", "channel"]].replace({"Unknown": "Not supplied", "web": "online"})
# rename columns and drop rows/columns in a demonstration copy.
renamed_demo = clean.rename(columns={"net_amount": "sales_amount", "order_date": "sale_date"})
small_demo = clean.head(3).drop(columns=["email"], errors="ignore")
dropped_rows_demo = clean.drop(index=clean.index[:1])
print("isnull == isna:", missing_isnull.equals(missing_isna))
print("notnull == notna:", observed_notnull.equals(observed_notna))
print("replace:"); display(replace_demo.head())
print("rename/drop:", renamed_demo.columns[:4].tolist(), small_demo.shape, dropped_rows_demo.shape)


### Part 6 — Manipulation and transformation

Demonstrates adding/updating/deleting a column, `insert`, Series `apply`/`map`, `DataFrame.map`, `assign`, `where`, `mask`, lambdas, and vectorized arithmetic. `DataFrame.map` is available in modern pandas; the guarded fallback keeps the example usable on older Colab runtimes.

In [ ]:
transform_demo = clean[["order_id", "quantity", "unit_price", "net_amount", "status"]].copy()
# Adding and updating columns; insert places a column at a requested position.
transform_demo["tax_estimate"] = transform_demo["net_amount"] * 0.05
transform_demo["status"] = transform_demo["status"].str.title()
transform_demo.insert(1, "order_id_length", transform_demo["order_id"].str.len())
# map maps each value; apply applies a function to each Series value.
transform_demo["status_code"] = transform_demo["status"].map({"Shipped": 1, "Delivered": 2, "Cancelled": 0}).fillna(-1)
transform_demo["amount_rounded"] = transform_demo["net_amount"].apply(lambda value: round(value, 0))
# DataFrame.map transforms every scalar in the selected frame.
scalar_demo = transform_demo[["quantity", "unit_price"]].head().copy()
if hasattr(scalar_demo, "map"):
    scalar_demo_mapped = scalar_demo.map(lambda value: value * 2 if pd.notna(value) else value)
else:  # compatibility with pandas releases before DataFrame.map
    scalar_demo_mapped = scalar_demo.applymap(lambda value: value * 2 if pd.notna(value) else value)
# assign returns a new DataFrame; where keeps values meeting condition; mask replaces values meeting condition.
assigned_demo = transform_demo.assign(net_after_tax=lambda frame: frame["net_amount"] + frame["tax_estimate"])
where_demo = transform_demo["net_amount"].where(transform_demo["net_amount"] >= 1000, 0)
mask_demo = transform_demo["net_amount"].mask(transform_demo["net_amount"] < 1000, 0)
# Vectorized calculation (no row-wise Python loop).
transform_demo["recalculated"] = transform_demo["quantity"] * transform_demo["unit_price"] * (1 - 0.01 * clean["discount_pct"].to_numpy())
# Deleting a column with drop().
without_tax = transform_demo.drop(columns=["tax_estimate"])
display(transform_demo.head()); display(scalar_demo_mapped); print(assigned_demo.shape, where_demo.head().tolist(), mask_demo.head().tolist(), without_tax.shape)


### Part 7 — Sorting and ranking

Sort rows by values or index, calculate rank, and select largest/smallest records.

In [ ]:
sorted_values = clean.sort_values(["net_amount", "order_date"], ascending=[False, True])
sorted_index = clean.set_index("order_id").sort_index()
ranked = clean[["order_id", "net_amount"]].copy()
ranked["sales_rank"] = ranked["net_amount"].rank(method="dense", ascending=False).astype(int)
display(sorted_values[["order_id", "net_amount"]].head())
display(ranked.sort_values("sales_rank").head())
display(clean.nlargest(5, "net_amount")[["order_id", "net_amount"]])
display(clean.nsmallest(5, "net_amount")[["order_id", "net_amount"]])


### Part 8 — GroupBy and aggregation

Grouping by one or multiple columns, standard and named aggregations, `transform`, group filtering, custom aggregation, and `reset_index`.

In [ ]:
# One-column grouping and multi-column grouping.
by_status = clean.groupby("status")["net_amount"].sum()
by_category_channel = clean.groupby(["category", "channel_label"], as_index=False).agg(
    order_count=("order_id", "nunique"), total_sales=("net_amount", "sum"),
    average_sales=("net_amount", "mean"), median_sales=("net_amount", "median")
)
# Standard aggregation functions and named aggregation.
standard_agg = clean.groupby("category")["net_amount"].agg(["count", "sum", "mean", "min", "max", "std"])
named_agg = clean.groupby("category").agg(total_sales=("net_amount", "sum"), units_sold=("quantity", "sum"))
# transform returns a result aligned to each original row (category-level average).
clean["category_avg_order"] = clean.groupby("category")["net_amount"].transform("mean")
# GroupBy.filter keeps whole groups that meet a condition.
large_categories = clean.groupby("category").filter(lambda group: group["order_id"].nunique() >= 10)
# Custom aggregation and reset_index.
range_agg = clean.groupby("category")["net_amount"].agg(lambda values: values.max() - values.min()).rename("sales_range")
summary_reset = clean.groupby("category")["net_amount"].sum().reset_index(name="total_sales")
display(by_status); display(by_category_channel); display(standard_agg); display(named_agg); display(range_agg); display(summary_reset)
print("Rows in groups with >=10 orders:", len(large_categories))


### Part 9 — Combining datasets

`concat` stacks or aligns objects. `merge` handles key-based joins, including multiple keys and self-joins. The previous join section demonstrates inner/left/right/outer joins and `DataFrame.join()`.

In [ ]:
# concat: append rows (ignore_index) and combine columns side-by-side (axis=1).
concat_rows = pd.concat([clean.head(3), clean.iloc[3:5]], ignore_index=True)
concat_columns = pd.concat([clean[["order_id"]].head(), clean[["net_amount"]].head()], axis=1)
# Merge using multiple columns, illustrated with a small grouped lookup.
monthly_lookup = clean.assign(month_key=clean["order_date"].dt.to_period("M").astype("string")).groupby(["customer_id", "month_key"], as_index=False).agg(month_spend=("net_amount", "sum"))
orders_month = clean.assign(month_key=clean["order_date"].dt.to_period("M").astype("string"))
multiple_key_merge = orders_month.merge(monthly_lookup, on=["customer_id", "month_key"], how="left", validate="many_to_one")
# Self join: pair customers with other customers in the same city (small sample to limit result size).
people = clean[["customer_id", "customer_name", "city"]].drop_duplicates("customer_id").dropna(subset=["city"]).head(12)
self_join = people.merge(people, on="city", suffixes=("_a", "_b"))
self_join = self_join.loc[self_join["customer_id_a"] < self_join["customer_id_b"]]
# index-based join (the prior section also performs this against the full orders).
index_join_demo = clean.head().set_index("customer_id").join(customers.set_index("customer_id")[["segment"]], rsuffix="_ref")
print("concat:", concat_rows.shape, concat_columns.shape, "multi-key merge:", multiple_key_merge.shape, "self join:", self_join.shape, "index join:", index_join_demo.shape)
display(self_join.head())


### Part 10 — Date and time operations

Parse datetimes, extract year/month/day/weekday, filter a date range, calculate date differences, generate a date range, localize/convert time zones, resample, and demonstrate rolling, shift, and differences.

In [ ]:
date_demo = clean[["order_id", "order_date", "net_amount"]].copy()
date_demo["order_date"] = pd.to_datetime(date_demo["order_date"], errors="coerce")
date_demo["year"] = date_demo["order_date"].dt.year
date_demo["month"] = date_demo["order_date"].dt.month
date_demo["day"] = date_demo["order_date"].dt.day
date_demo["weekday"] = date_demo["order_date"].dt.day_name()
start_date, end_date = date_demo["order_date"].min(), date_demo["order_date"].max()
date_filtered = date_demo.loc[date_demo["order_date"].between(start_date, end_date)]
date_demo["days_from_first_order"] = (date_demo["order_date"] - start_date).dt.days
calendar_demo = pd.date_range(start=start_date, end=end_date, freq="D", name="calendar_date")
# Time zone localization and conversion example.
localized_demo = date_demo["order_date"].dt.tz_localize("Asia/Kolkata", ambiguous="NaT", nonexistent="NaT")
converted_demo = localized_demo.dt.tz_convert("UTC")
# Resample, rolling, shift, diff, and percent change on the daily sales series.
daily_demo_sales = clean.set_index("order_date")["net_amount"].resample("D").sum().sort_index()
daily_features = pd.DataFrame({"sales": daily_demo_sales})
daily_features["rolling_7_day_mean"] = daily_features["sales"].rolling(window=7, min_periods=1).mean()
daily_features["previous_day_sales"] = daily_features["sales"].shift(1)
daily_features["change_from_previous_day"] = daily_features["sales"].diff()
daily_features["pct_change"] = daily_features["sales"].pct_change(fill_method=None)
print("Calendar days:", len(calendar_demo), "| timezone:", converted_demo.dt.tz)
display(date_demo.head()); display(daily_features.head(10))


### Part 11 — String operations and regular expressions

The main cleaning section already uses `lower`, `upper`, `title`, `strip`, `replace`, `contains`, `extract`, `len`, and regex. Here are explicit `startswith`, `endswith`, and `split` examples too.

In [ ]:
string_demo = clean[["customer_name", "email", "city"]].copy()
string_demo["name_lower"] = string_demo["customer_name"].str.lower()
string_demo["city_upper"] = string_demo["city"].str.upper()
string_demo["name_title"] = string_demo["customer_name"].str.title()
string_demo["name_stripped"] = string_demo["customer_name"].str.strip()
string_demo["email_no_spaces"] = string_demo["email"].str.replace(r"\s+", "", regex=True)
string_demo["email_contains_example"] = string_demo["email"].str.contains("example", case=False, na=False)
string_demo["name_starts_a"] = string_demo["customer_name"].str.startswith("A", na=False)
string_demo["email_ends_com"] = string_demo["email"].str.endswith(".com", na=False)
string_demo["name_parts"] = string_demo["customer_name"].str.split(" ")
string_demo["first_name"] = string_demo["customer_name"].str.extract(r"^([^ ]+)", expand=False)
string_demo["name_length"] = string_demo["customer_name"].str.len()
string_demo["email_user"] = string_demo["email"].str.extract(r"^([^@]+)@", expand=False)
display(string_demo.head())


### Part 12 — Statistical analysis

Compute mean, median, mode, sum, count, min/max, standard deviation, variance, quantiles, correlation, covariance, cumulative sum/product, percentage change, and difference. Correlation/covariance describe associations and do not establish causation.

In [ ]:
numeric = clean[["quantity", "unit_price", "discount_pct", "gross_amount", "net_amount"]].apply(pd.to_numeric, errors="coerce")
stats = pd.DataFrame({
    "mean": numeric.mean(), "median": numeric.median(), "sum": numeric.sum(),
    "count": numeric.count(), "min": numeric.min(), "max": numeric.max(),
    "std": numeric.std(), "variance": numeric.var(),
    "q25": numeric.quantile(0.25), "q75": numeric.quantile(0.75),
})
mode_values = numeric.mode().head()
correlation = numeric.corr()
covariance = numeric.cov()
# Cumulative and period-to-period statistics on chronologically sorted daily sales.
sales_series = daily_demo_sales.sort_index()
cumulative_sum = sales_series.cumsum()
cumulative_product_demo = (1 + sales_series.pct_change(fill_method=None).fillna(0)).cumprod()
period_pct_change = sales_series.pct_change(fill_method=None)
period_difference = sales_series.diff()
display(stats); print("Modes:"); display(mode_values); print("Correlation:"); display(correlation); print("Covariance:"); display(covariance)
display(pd.DataFrame({"sales": sales_series, "cumsum": cumulative_sum, "cumprod_of_growth": cumulative_product_demo, "pct_change": period_pct_change, "diff": period_difference}).head(10))


### Extra topics — useful for a stronger project

`cut` and `qcut` bin continuous values; `explode` expands list-like values into rows; `pipe` makes multi-step transformations easier to compose; `memory_usage` helps inspect data size.

In [ ]:
extra_demo = clean[["order_id", "net_amount", "customer_name"]].copy()
extra_demo["fixed_bins"] = pd.cut(extra_demo["net_amount"], bins=4, labels=["Band 1", "Band 2", "Band 3", "Band 4"])
extra_demo["quantile_bins"] = pd.qcut(extra_demo["net_amount"].rank(method="first"), q=4, labels=["Q1", "Q2", "Q3", "Q4"])
name_tokens = extra_demo.assign(name_token=extra_demo["customer_name"].str.split()).explode("name_token")
def add_revenue_per_unit(frame):
    return frame.assign(revenue_per_unit=frame["net_amount"] / frame["quantity"])
piped_demo = clean.pipe(add_revenue_per_unit)
print("Memory usage (bytes):", int(clean.memory_usage(deep=True).sum()))
display(extra_demo.head()); display(name_tokens[["order_id", "name_token"]].head()); display(piped_demo[["order_id", "revenue_per_unit"]].head())


## 17. What this project demonstrates

- Profiling messy data before cleaning and quantifying changes.
- Normalizing text and validating emails with vectorized `.str` operations.
- Coercing invalid numbers/dates safely and handling missing values with documented choices.
- Removing duplicates by a declared business key and enforcing simple quality rules.
- Calculating measures with vectorized arithmetic; using `np.select`, `map`, `lambda`, and `apply`.
- Extracting calendar fields, generating a date range, localizing and converting time zones, and resampling.
- Applying inner, left, right, outer, cross, index-based, and as-of joins.
- Aggregating, pivoting, melting, encoding categories, reviewing outliers, and exporting CSV/Excel.

### Official references
- [Pandas user guide: missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [Pandas user guide: text data](https://pandas.pydata.org/docs/user_guide/text.html)
- [Pandas user guide: time series/date functionality](https://pandas.pydata.org/docs/user_guide/timeseries.html)
- [Pandas merge and join reference](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html)
